In [ ]:
import pandas as pd
import numpy as np
import glob

print("=== Telangana PDS Cleaning Pipeline Started ===")

# =========================
# 1. LOAD TRANSACTIONS
# =========================
transaction_path = r"../data/raw/transactions"
transaction_files = glob.glob(transaction_path + r"/**/*.csv", recursive=True)

transaction_dfs = []

for file in transaction_files:
    df = pd.read_csv(file)
    year = file.split("\\")[-2]
    df["year"] = year
    transaction_dfs.append(df)

transactions = pd.concat(transaction_dfs, ignore_index=True)

print("Transactions Loaded:", transactions.shape)

# =========================
# 2. LOAD CARD STATUS
# =========================
card_path = r"../data/raw/card_status"
card_files = glob.glob(card_path + r"/**/*.csv", recursive=True)

card_dfs = []

for file in card_files:
    df = pd.read_csv(file)
    year = file.split("\\")[-2]
    df["year"] = year
    card_dfs.append(df)

card_status = pd.concat(card_dfs, ignore_index=True)

print("Card Status Loaded:", card_status.shape)

# =========================
# 3. LOAD FPS LOCATION
# =========================
fps_files = glob.glob("../data/raw/fps_locations/*.csv")
fps_locations = pd.read_csv(fps_files[0])

print("FPS Loaded:", fps_locations.shape)

# =========================
# 4. CLEAN MISSING VALUES
# =========================
fps_locations["address"] = fps_locations["address"].fillna("Unknown")

fps_locations["longitude"] = fps_locations["longitude"].fillna(
    fps_locations["longitude"].median()
)

fps_locations["latitude"] = fps_locations["latitude"].fillna(
    fps_locations["latitude"].median()
)

# =========================
# 5. REMOVE DUPLICATES
# =========================
transactions = transactions.drop_duplicates()
card_status = card_status.drop_duplicates()
fps_locations = fps_locations.drop_duplicates()

# =========================
# 6. STANDARDIZE COLUMN NAMES
# =========================
transactions.columns = transactions.columns.str.strip().str.lower()
card_status.columns = card_status.columns.str.strip().str.lower()
fps_locations.columns = fps_locations.columns.str.strip().str.lower()

# =========================
# 7. VERIFY KEYS
# =========================
print("Common keys (transactions & card_status):")
print(set(transactions.columns).intersection(card_status.columns))

print("Common keys (transactions & fps_locations):")
print(set(transactions.columns).intersection(fps_locations.columns))

# =========================
# 8. MERGE DATASETS
# =========================
master_df = pd.merge(
    transactions,
    card_status,
    on=["shopno", "distcode", "year"],
    how="left"
)

master_df = pd.merge(
    master_df,
    fps_locations,
    on=["shopno", "distcode"],
    how="left"
)

print("MASTER DATASET CREATED:", master_df.shape)

# =========================
# 9. FINAL CHECK
# =========================
print(master_df.head())

print("\nMissing Values:")
print(master_df.isnull().sum().sort_values(ascending=False).head(10))

# =========================
# 10. SAVE DATA
# =========================
master_df.to_csv("../data/processed/master_pds_data.csv", index=False)

print("\nSaved: master_pds_data.csv")
print("=== CLEANING COMPLETED SUCCESSFULLY ===")

Transactions Loaded: (482407, 19)
Card Status Loaded: (517516, 28)
FPS Loaded: (17434, 11)


distCode       0
distName       0
officeCode     0
officeName     0
shopNo         0
address       12
longitude     66
latitude      66
fpsStatus      0
fpsType        0
dateTime       0
dtype: int64